In [ ]:
from utils.motif import init_repro

init_repro(42, deterministic=True)

import sys
sys.path.append("./utils")

import numpy as np
import torch
import wandb
import time
import os
import pickle
import clip

from transformers import (
    CLIPProcessor,
    CLIPModel,
    CLIPVisionModelWithProjection,
    CLIPTokenizer,
    CLIPTextModelWithProjection,
    AutoTokenizer,
    AutoModel,
    AutoProcessor,
)

from utils.video_embedder import VideoEmbedder, Create_Concepts
from utils.motif import MoTIF, CBMTransformer, mean_cbm
from utils.explanations import explain_instance
import core.vision_encoder.pe as pe
import core.vision_encoder.transforms as pe_transformer

In [ ]:
# Training hyperparameters
num_epochs = 100
batch_size = 32
lse_tau = 1.0
l1_lambda = 1e-3
lambda_sparse = 1e-3
lr = 1e-3
transformer_layers = 1
diagonal_attention = True
enforce_nonneg = True
class_weights = True
weight_decay = 1e-2
d = 1

# Dataset settings
test_split = "s1"
window_size = 32
dataset = "breakfast"
random = True
use_wandb = True
clip_model = "res50"

# Map dataset name
dataset_map = {
    "breakfast": "Breakfast",
    "ucf101": "UCF101",
    "hmdb51": "HMDB",
    "something2": "Something2"
}
dataset_name = dataset_map.get(dataset, dataset)

folder_path = [f"../Datasets/{dataset_name}/Video_data"]
output_dir = "../Embeddings/Datasets"

In [ ]:
# Load CLIP model and processor
if clip_model == "b32":
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval()
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", use_fast=False)
    embedd_path = f"../Embeddings/Videos/{dataset_name}/{random}_{window_size}_clip_b32.pkl"
    clip_name = "clip"
elif clip_model == "b16":
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").eval()
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16", use_fast=False)
    embedd_path = f"../Embeddings/Videos/{dataset_name}/{random}_{window_size}_clip_b16.pkl"
    clip_name = "clip"
elif clip_model == "l14":
    model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").eval()
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14", use_fast=False)
    embedd_path = f"../Embeddings/Videos/{dataset_name}/{random}_{window_size}_clip_l14.pkl"
    clip_name = "clip"
elif clip_model == "res50":
    model, preprocess = clip.load("RN50", device="cpu")
    processor = preprocess
    embedd_path = f"../Embeddings/Videos/{dataset_name}/{random}_{window_size}_clip_res50.pkl"
    clip_name = "res50"
elif clip_model == "clip4clip":
    model = CLIPVisionModelWithProjection.from_pretrained("Searchium-ai/clip4clip-webvid150k").eval()
    model_text = CLIPTextModelWithProjection.from_pretrained("Searchium-ai/clip4clip-webvid150k")
    processor = CLIPTokenizer.from_pretrained("Searchium-ai/clip4clip-webvid150k")
    embedd_path = f"../Embeddings/Videos/{dataset_name}/{random}_{window_size}_clip_clip4clip.pkl"
    clip_name = "clip4clip"
elif clip_model == "siglip":
    model = AutoModel.from_pretrained("google/siglip-base-patch16-224")
    processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
    embedd_path = f"../Embeddings/Videos/{dataset_name}/{random}_{window_size}_clip_siglip.pkl"
    clip_name = "siglip"
elif clip_model == "siglipl14":
    model = AutoModel.from_pretrained("google/siglip-so400m-patch14-384")
    processor = AutoProcessor.from_pretrained("google/siglip-so400m-patch14-384")
    embedd_path = f"../Embeddings/Videos/{dataset_name}/{random}_{window_size}_clip_siglipl14.pkl"
    clip_name = "siglipl14"
elif clip_model == "pe-l14":
    model = pe.CLIP.from_config("PE-Core-L14-336", pretrained=True)
    processor = pe_transformer.get_image_transform(model.image_size)
    tokenizer = pe_transformer.get_text_tokenizer(model.context_length)
    clip_name = "pe-l14"
    embedd_path = f"../Embeddings/Videos/{dataset_name}/{random}_{window_size}_clip_pe-l14.pkl"

else:
    model = None
    processor = None
    model_text = None

# Initialize embedder
embedder = VideoEmbedder(clip_name, model, processor)
embedder.dataset_name = dataset

# Load or create embeddings
if os.path.exists(embedd_path):
    with open(embedd_path, 'rb') as f:
        embedder = pickle.load(f)
    print(f"Loaded existing embedder from {embedd_path}")
else:
    embedder.process_data(
        folder_path,
        window_size=window_size,
        output_path=output_dir,
        save_intermediate=False,
    )
    with open(embedd_path, "wb") as f:
        pickle.dump(embedder, f)

# Initialize concepts
if clip_model == "clip4clip":
    concepts = Create_Concepts(clip_name, model_text, processor)
elif clip_model == "pe-l14":
    concepts = Create_Concepts(clip_name, model, tokenizer)
else:
    concepts = Create_Concepts(clip_name, model, processor)


In [ ]:

def create_artificial_concept_embeddings(dim=1024, num_concepts=None, seed=42):
    """Create concept embeddings (K, dim) representing temporal patterns."""
    rng = np.random.default_rng(seed)
    
    # Comprehensive list of temporal pattern concepts
    all_concept_names = [
        # Basic trends
        "ascending", "descending", "constant",
        
        # Curved patterns
        "u_shaped", "inverted_u", "s_shaped", "inverted_s",
        
        # Rate of change
        "increasing_rate", "decreasing_rate", "accelerating", "decelerating",
        
        # Periodic patterns
        "periodic", "periodic_fast", "periodic_slow", "periodic_phase_shift",
        
        # Step functions
        "step_up", "step_down", "step_up_down", "step_down_up",
        
        # Exponential patterns
        "exponential_growth", "exponential_decay", "logarithmic",
        
        # Multi-peak patterns
        "double_peak", "triple_peak", "sawtooth", "square_wave",
        
        # Asymmetric patterns
        "fast_rise_slow_fall", "slow_rise_fast_fall", "plateau_start", "plateau_end",
        
        # Complex patterns
        "oscillating_decay", "oscillating_growth", "chirp", "spike",
        
        # Edge cases
        "random", "noise", "sparse"
    ]
    
    # Use all concepts if num_concepts is None, otherwise use first N
    if num_concepts is None:
        concept_names = all_concept_names
    else:
        concept_names = all_concept_names[:num_concepts]
    
    concept_embeddings = []
    
    for i, name in enumerate(concept_names):
        x = np.linspace(0, 1, dim)
        t = np.linspace(0, 4 * np.pi, dim)
        
        # Basic trends
        if name == "ascending":
            emb = np.linspace(-0.5, 0.5, dim)
        elif name == "descending":
            emb = np.linspace(0.5, -0.5, dim)
        elif name == "constant":
            emb = np.ones(dim) * 0.1
        
        # Curved patterns
        elif name == "u_shaped":
            mid = dim // 2
            emb = np.concatenate([
                np.linspace(0.5, -0.5, mid),
                np.linspace(-0.5, 0.5, dim - mid)
            ])
        elif name == "inverted_u":
            mid = dim // 2
            emb = np.concatenate([
                np.linspace(-0.5, 0.5, mid),
                np.linspace(0.5, -0.5, dim - mid)
            ])
        elif name == "s_shaped":
            emb = 0.5 * (np.tanh(5 * (x - 0.5)) + 1) - 0.5
        elif name == "inverted_s":
            emb = -0.5 * (np.tanh(5 * (x - 0.5)) + 1) + 0.5
        
        # Rate of change
        elif name == "increasing_rate":
            emb = 0.5 * x ** 2 - 0.25
        elif name == "decreasing_rate":
            emb = -0.5 * x ** 2 + 0.25
        elif name == "accelerating":
            emb = 0.4 * x ** 3 - 0.2
        elif name == "decelerating":
            emb = -0.4 * x ** 3 + 0.2
        
        # Periodic patterns
        elif name == "periodic":
            emb = 0.3 * np.sin(t)
        elif name == "periodic_fast":
            emb = 0.3 * np.sin(2 * t)
        elif name == "periodic_slow":
            emb = 0.3 * np.sin(0.5 * t)
        elif name == "periodic_phase_shift":
            emb = 0.3 * np.sin(t + np.pi / 2)
        
        # Step functions
        elif name == "step_up":
            emb = np.where(x < 0.5, -0.3, 0.3)
        elif name == "step_down":
            emb = np.where(x < 0.5, 0.3, -0.3)
        elif name == "step_up_down":
            emb = np.where(x < 0.33, -0.3, np.where(x < 0.67, 0.3, -0.3))
        elif name == "step_down_up":
            emb = np.where(x < 0.33, 0.3, np.where(x < 0.67, -0.3, 0.3))
        
        # Exponential patterns
        elif name == "exponential_growth":
            emb = 0.3 * (np.exp(2 * x) - 1) / (np.exp(2) - 1) - 0.15
        elif name == "exponential_decay":
            emb = 0.3 * (np.exp(-2 * x)) - 0.15
        elif name == "logarithmic":
            emb = 0.3 * np.log1p(10 * x) / np.log(11) - 0.15
        
        # Multi-peak patterns
        elif name == "double_peak":
            emb = 0.3 * (np.sin(2 * t) + 0.5 * np.sin(4 * t))
        elif name == "triple_peak":
            emb = 0.25 * (np.sin(3 * t) + 0.5 * np.sin(6 * t))
        elif name == "sawtooth":
            emb = 0.3 * (2 * (t / (2 * np.pi) % 1) - 1)
        elif name == "square_wave":
            emb = 0.3 * np.sign(np.sin(t))
        
        # Asymmetric patterns
        elif name == "fast_rise_slow_fall":
            rise = 0.4 * np.exp(5 * x[:dim//2]) / np.exp(2.5) - 0.2
            fall = 0.4 * np.exp(-2 * (x[dim//2:] - 0.5)) - 0.2
            emb = np.concatenate([rise, fall])
        elif name == "slow_rise_fast_fall":
            rise = 0.4 * np.exp(2 * x[:dim//2]) / np.exp(1) - 0.2
            fall = 0.4 * np.exp(-5 * (x[dim//2:] - 0.5)) - 0.2
            emb = np.concatenate([rise, fall])
        elif name == "plateau_start":
            plateau_len = int(0.3 * dim)
            emb = np.concatenate([np.full(plateau_len, 0.2), np.linspace(0.2, -0.2, dim - plateau_len)])
        elif name == "plateau_end":
            plateau_len = int(0.7 * dim)
            emb = np.concatenate([np.linspace(-0.2, 0.2, plateau_len), np.full(dim - plateau_len, 0.2)])
        
        # Complex patterns
        elif name == "oscillating_decay":
            emb = 0.3 * np.sin(2 * t) * np.exp(-x)
        elif name == "oscillating_growth":
            emb = 0.3 * np.sin(2 * t) * (1 - np.exp(-x))
        elif name == "chirp":
            freq = 0.5 + 2 * x
            emb = 0.3 * np.sin(2 * np.pi * freq * x)
        elif name == "spike":
            emb = 0.5 * np.exp(-((x - 0.5) ** 2) / 0.01) - 0.25
        
        # Edge cases
        elif name == "random":
            emb = rng.normal(0, 0.3, dim)
        elif name == "noise":
            emb = rng.normal(0, 0.1, dim)
        elif name == "sparse":
            emb = rng.choice([-0.5, 0, 0.5], size=dim, p=[0.1, 0.8, 0.1])
        else:
            # Default: random pattern
            emb = rng.normal(0, 0.3, dim)
        
        if name != "noise" and name != "random":
            emb = emb + rng.normal(0, 0.05, dim)
        
        emb = emb / (np.linalg.norm(emb) + 1e-9)
        concept_embeddings.append(emb)
    
    return np.array(concept_embeddings), concept_names


def generate_order_sensitive_data(samples, dim=1024, num_classes=5, seed=42, pattern_strength=0.5, shuffled=False):
    """Generate synthetic sequences with order-dependent temporal patterns."""
    rng = np.random.default_rng(seed)
    X = {}
    y = []

    data_keys = list(samples.keys())
    num_pattern_dims = max(1, int(dim * pattern_strength))  # Use multiple dimensions for patterns
    
    # Class descriptions:
    # class_0: Ascending linear pattern (monotonically increasing from -1 to 1)
    # class_1: Descending linear pattern (monotonically decreasing from 1 to -1)
    # class_2: U-shaped pattern (decreasing then increasing, valley shape)
    # class_3: Inverted U-shaped pattern (increasing then decreasing, peak shape)s
    # class_4: Sinusoidal pattern (periodic oscillation with sin function)

    for i, key in enumerate(data_keys):
        n = len(samples[key])
        seq = rng.uniform(-1, 1, (n, dim))
        t = np.linspace(0, 2 * np.pi, n)
        
        if i % num_classes == 0:
            pattern = np.linspace(-1, 1, n)
            label = "class_0"
        elif i % num_classes == 1:
            pattern = np.linspace(1, -1, n)
            label = "class_1"
        elif i % num_classes == 2:
            mid = n // 2
            pattern = np.concatenate([
                np.linspace(1, -1, mid),
                np.linspace(-1, 1, n - mid)
            ])
            label = "class_2"
        elif i % num_classes == 3:
            mid = n // 2
            pattern = np.concatenate([
                np.linspace(-1, 1, mid),
                np.linspace(1, -1, n - mid)
            ])
            label = "class_3"
        else:
            pattern = np.sin(t)
            label = "class_4"

        for dim_idx in range(num_pattern_dims):
            amplitude = 0.5 + 0.5 * (dim_idx % 3) / 3
            seq[:, dim_idx] = pattern * amplitude + 0.1 * rng.normal(0, 1, n)
        
        seq = seq + rng.normal(0, 0.1, (n, dim))
        
        if shuffled:
            seq = np.random.permutation(seq)
        
        X[key] = seq.astype(np.float32)
        y.append(label)

    return X, np.array(y)


# Generate synthetic sequences with temporal patterns
X, y = generate_order_sensitive_data(
    embedder.video_embeddings,
    dim=1024,
    num_classes=5,
    seed=42,
    pattern_strength=0.3,
    shuffled = False,
)

# Create concept embeddings for temporal patterns
concept_embeddings, concept_names = create_artificial_concept_embeddings(
    dim=1024,
    num_concepts=None,
    seed=42
)

print(f"Generated {len(X)} synthetic samples")
print(f"Sequence shape: {list(X.values())[0].shape}")
print(f"Concept embeddings: {concept_embeddings.shape}")
print(f"Number of concepts: {len(concept_names)}")
print(f"Label distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

# Replace embedder data with synthetic data
embedder.video_embeddings = X
embedder.labels = y

# Set synthetic concept embeddings
concepts.text_embeddings = torch.tensor(concept_embeddings, dtype=torch.float32)
concepts.text_concepts = concept_names

In [ ]:
# Initialize and train MoTIF
cbm_model = MoTIF(embedder, concepts)
cbm_model.preprocess(dataset, info=test_split)

if use_wandb:
    time_now = time.strftime("%Y-%m-%d_%H-%M-%S", time.localtime())
    name = f"{embedder.dataset_name}_clip_basic_{time_now}"
    wandb_run = wandb.init(project="motif", name=name)
    cbm_model.zero_shot(concepts, wandb_run=wandb_run)
    wandb_run.log({
        "backbone": clip_name,
        "window_size": window_size,
        "dataset": dataset,
        "random": random,
        "test_split": test_split,
        "d": d,
    })
else:
    wandb_run = None
    cbm_model.zero_shot(concepts)

cbm_model.model = CBMTransformer(
    cbm_model.num_concepts,
    num_classes=cbm_model.num_classes,
    transformer_layers=transformer_layers,
    lse_tau=lse_tau,
    dimension=d,
    diagonal_attention=diagonal_attention,
)

cbm_model.train_model(
    num_epochs=num_epochs,
    l1_lambda=l1_lambda,
    lambda_sparse=lambda_sparse,
    lr=lr,
    batch_size=batch_size,
    enforce_nonneg=enforce_nonneg,
    class_weights=class_weights,
    wandb_run=wandb_run,
)

mean_cbm(cbm_model, wandb_run=wandb_run)

if use_wandb:
    wandb_run.finish()

In [ ]:
# Evaluate temporal learning: compare accuracy on original vs shuffled sequences
results = {}
for trial in range(5):
    correct_original = 0
    correct_shuffled = 0
    
    for i in range(len(cbm_model.X_test)):
        video = cbm_model.X_test[i]
        true_idx = cbm_model.y_test[i]
        
        res = explain_instance(cbm_model.model, video)
        class_pred = res["target_class"]
        
        shuffled_video = torch.from_numpy(np.random.permutation(video)).to("cuda:0")
        logits, _, _, _ = cbm_model.model(shuffled_video)
        pred_shuffled = int(logits.argmax(dim=1).item())
        
        if true_idx == class_pred:
            correct_original += 1
        if true_idx == pred_shuffled:
            correct_shuffled += 1
    
    acc_original = correct_original / len(cbm_model.X_test)
    acc_shuffled = correct_shuffled / len(cbm_model.X_test)
    
    print(f"Trial {trial+1}: Original={acc_original:.4f}, Shuffled={acc_shuffled:.4f}")
    results[trial] = [acc_original, acc_shuffled]

print(f"\nAverage: Original={np.mean([r[0] for r in results.values()]):.4f}, "
      f"Shuffled={np.mean([r[1] for r in results.values()]):.4f}")